# Generating Test Datasets

We're building an eval for a prompt that helps users write **AWS-specific** output in one of three formats — **Python code, JSON config, or a regex** — with *no* extra explanation, headers, or footers.

**The prompt under test (version 1):**

```python
prompt = f"""
Please provide a solution to the following task:
{task}
"""
```

This notebook covers **step 2** of the eval workflow: building the eval dataset. Rather than hand-write test cases, we ask Claude to generate them — a perfect job for a fast model like **Haiku**.

> ⚠️ **Two adaptations from the course** (models moved on since it was recorded):
> 1. **Assistant prefilling** (`add_assistant_message(messages, "```json")`) returns a **400** on flagship models. The course technique needs it, so this notebook runs on **`claude-haiku-4-5-20251001`** — which still supports prefilling *and* is the model the course recommends for dataset generation.
> 2. The course reads `response.content[0].text`. We use our **`get_text()`** helper instead, which scans for the text block (dodges thinking-block errors).

## Setup — client, model, helpers

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())   # find the repo-root .env from any folder depth

import json
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"   # fast + supports assistant prefilling


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return get_text(message)


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Generate the dataset with Claude

The dataset is an array of JSON objects, each with a `"task"` property describing something solvable by a single Python function, a single JSON object, or a single regex. We steer Claude to return raw JSON using the **prefill + stop-sequence** trick from the last section: prefill ` ```json ` so it thinks the code block is already open, and stop on ` ``` ` before it closes the fence.

In [2]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")   # prefill: pretend the code block is open
    text = chat(messages, stop_sequences=["```"])  # stop before Claude closes the fence
    return json.loads(text)

## Test it

Run the generator and inspect the three test cases — a mix of Python, JSON, and regex AWS tasks.

In [3]:
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL (e.g., 's3.us-west-2.amazonaws.com'). The function should return the region code."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'."},
 {'task': "Write a regex pattern that matches valid AWS EC2 instance IDs (format: i-followed by 8 or 17 hexadecimal characters, e.g., 'i-1234567890abcdef0')."}]

## Save the dataset

Persist it so the next lesson (`Running the eval`) can load it back without regenerating.

> **Path note:** the course writes `dataset.json` "in the same directory as your notebook." Our `.vscode/settings.json` pins the kernel's working directory to the **repo root** (so `.env` loads from anywhere), which means a bare `open('dataset.json')` would land at the repo root. We write to an explicit section path instead, so the file co-locates with this notebook and the next lesson can find it.

In [4]:
DATASET_PATH = "01-building-with-the-claude-api/02-prompt-evaluation/dataset.json"

with open(DATASET_PATH, "w") as f:
    json.dump(dataset, f, indent=2)

print(f"Saved {len(dataset)} tasks to {DATASET_PATH}")

Saved 3 tasks to 01-building-with-the-claude-api/02-prompt-evaluation/dataset.json
